# 03. Invoking a Previously Created Prompt Agent

**⚠️ Dependency:** this script does **not** create an agent. It **invokes an agent that must already exist**
in your Foundry project — specifically `"IT-HelpDesk-Agent"`, the same name created in `02_prompt_agent.py`
(and later re-versioned in `05_IT_HelpDesk_agent.py`). Run `02_prompt_agent.py` first (or make sure that agent
name already exists in your project) before running this one, or the call will fail with a "not found" style
error.

Instead of the Foundry-native SDK surface, this script calls `client.get_openai_client()` to obtain an
**OpenAI-SDK-compatible client** pointed at your Foundry project, then invokes the agent through the
**Responses API** using an `extra_body={"agent_reference": {...}}` parameter. This is the pattern you'll use
whenever you want to talk to a Foundry agent using familiar OpenAI SDK calls rather than the
`azure-ai-projects`/`azure-ai-agents` native surface.

Note this call references the agent **by name only**, with no `version` specified — Foundry resolves that to
whichever version it considers current/latest at call time, which is a source of ambiguity worth understanding
(contrast with `07_helpdesk_client.py`, which pins an explicit `version`).

**Difficulty:** Beginner


## Prerequisites

**Pip packages:** `azure-ai-projects`, `azure-identity`, `python-dotenv` — all already in root `requirements.txt`.

**Azure resource(s) required:** The same Foundry project as `02_prompt_agent.py`, with the
`"IT-HelpDesk-Agent"` agent already created in it.

**Environment variables** (root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_AGENT_NAME` (optional, defaults to `"IT-HelpDesk-Agent"`)

Auth: Entra ID via `DefaultAzureCredential` (`az login`).


## What You'll Learn

- How `client.get_openai_client()` bridges `azure-ai-projects` to an OpenAI-SDK-shaped client
- How to invoke a named Foundry agent through the **Responses API** using `extra_body={"agent_reference": ...}`
- The difference between referencing an agent by name only (floating/latest) vs. by name **and** version (pinned)
- Why this script has a hard runtime dependency on an earlier script having already created the agent


### Setup and getting an OpenAI-compatible client

Same client construction as `02_prompt_agent.py`, but here we additionally call
`client.get_openai_client()`, which returns an object with the same method shapes as the standard `openai`
Python SDK (`.responses.create(...)`, `.conversations.create(...)`, etc.) — except requests are routed through
your Foundry project instead of api.openai.com.

- **💡 Exam tip:** Azure AI Foundry's Agent Service intentionally exposes an OpenAI-SDK-compatible surface (the
  **Responses API**) so that code written against OpenAI's SDK patterns can target Foundry-hosted agents with
  minimal changes — useful for migrating existing OpenAI Assistants/Responses code to Azure.
- **🔄 Alternatives:** You could instead use `azure-ai-agents`' native client methods directly (as seen in
  `11_azure_ai_foundry/05_hosted_agent`) for full access to Foundry-specific agent/thread management, rather
  than going through the OpenAI-compatible shim.


In [ ]:
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "IT-HelpDesk-Agent")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai = client.get_openai_client()


### Invoking the agent by name

`openai.responses.create(...)` is the standard OpenAI Responses API call, except `extra_body` carries Foundry's
`agent_reference` object — `{"name": AGENT_NAME, "type": "agent_reference"}` — telling the service "run this
input through the named agent's instructions/model/tools instead of a raw model call." No `version` key is
given, so Foundry uses its default resolution (typically the latest version) of `"IT-HelpDesk-Agent"`.
`response.output_text` is a convenience property that concatenates the agent's text output.

- **💡 Exam tip:** `extra_body` is the standard OpenAI-Python-SDK escape hatch for passing provider-specific
  fields the SDK's typed method signature doesn't know about yet — Azure uses it here to smuggle in the
  `agent_reference`. Recognize this pattern when you see `extra_body` in Azure AI Foundry sample code.
- **🔄 Alternatives:** Pin an explicit `"version"` inside `agent_reference` (as `07_helpdesk_client.py` does
  with `AGENT_VERSION = "2"`) for reproducible behavior instead of "whatever is latest right now."


In [ ]:
response = openai.responses.create(
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="How do I reset my company password?",
)

print(response.output_text)


## Summary

This notebook does not create anything — it sends one question ("How do I reset my company password?") to
the already-existing `IT-HelpDesk-Agent` via the OpenAI-compatible Responses API, using an `agent_reference`
to route the request to that named Foundry agent instead of a bare model. The printed output is the agent's
final text answer, generated using whatever `instructions` (and, once `05_IT_HelpDesk_agent.py` has run,
`tools`) that agent's current version carries.


## Try It Yourself

1. **Easy:** Change the `input` string to a VPN or software-install question and re-run.
2. **Intermediate:** Add an explicit `"version": "1"` to the `agent_reference` dict and compare the response
   to calling without a version — does behavior change once version 2 exists (after running
   `05_IT_HelpDesk_agent.py`)?
3. **Advanced:** Wrap this call in a loop that asks several questions in the same Python process and print
   `response.id` for each — investigate (via SDK docs) whether/how you'd thread these into one multi-turn
   `conversation` object, the way `07_helpdesk_client.py` does with `client.conversations.create()`.
